In [3]:
import pandas as pd

df = pd.read_csv("../train.csv")

In [4]:
print(df.shape)
print(df["Transported"].value_counts(normalize=True))
print(df.isna().sum())
df.head()

(8693, 14)
Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
        df, test_size = 0.2, stratify = df["Transported"], random_state=42
)

In [6]:
print(train_df.isna().sum().sort_values(ascending=False))

HomePlanet      169
ShoppingMall    169
CryoSleep       168
Cabin           161
Name            159
VIP             158
FoodCourt       154
Destination     153
Spa             150
RoomService     145
VRDeck          144
Age             139
PassengerId       0
Transported       0
dtype: int64


In [7]:
print(len(train_df), len(train_df.dropna()))

6954 5272


In [8]:
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Side"]

In [9]:
for row in (train_df, test_df):
    row[["Deck", "CabinNum", "Side"]] = row["Cabin"].str.split("/", expand=True)

In [10]:
drop_cols = ["PassengerId", "Name", "Cabin", "CabinNum"]

train_df = train_df.drop(columns = drop_cols)
test_df = test_df.drop(columns = drop_cols)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

y_train = train_df.pop("Transported").astype(int).values
y_test = test_df.pop("Transported").astype(int).values

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown  = "ignore", sparse_output=False)),
])

pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

X_train = pre.fit_transform(train_df)
X_test = pre.fit_transform(test_df)

print(X_train.shape, X_test.shape)

(6954, 26) (1739, 26)


In [12]:
import torch 
from torch.utils.data import Dataset

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32)


    def __len__(self):
        return len(self.X)


    def __getitem__(self,idx):
        return self.X[idx], self.y[idx]

In [13]:
from torch.utils.data import DataLoader 

train_ds = TabularDataset(X_train, y_train)
train_dl = DataLoader(train_ds, batch_size= 64, shuffle=True)
xb, yb = next(iter(train_dl))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)

torch.Size([64, 26]) torch.float32 torch.Size([64]) torch.float32


In [14]:
test_ds = TabularDataset(X_test, y_test)
test_dl = DataLoader(test_ds, batch_size = 64, shuffle = False)

In [15]:
print([v for v in dir() if not v.startswith("_")][-25:])

['df', 'drop_cols', 'exit', 'get_ipython', 'num_cols', 'num_pipe', 'open', 'pd', 'pre', 'quit', 'row', 't0', 'test_df', 'test_dl', 'test_ds', 'time', 'torch', 'train_df', 'train_dl', 'train_ds', 'train_test_split', 'xb', 'y_test', 'y_train', 'yb']


In [16]:
import numpy as np

X_train = train_ds.X.numpy() if hasattr(train_ds, "X") else np.array([train_ds[i][0].numpy() for i in range(len(train_ds))])
X_test = test_ds.X.numpy() if hasattr(test_ds, "X") else np.array([test_ds[i][0].numpy() for i in range(len(test_ds))])

In [17]:
print(X_train.shape, X_test.shape, np.isnan(X_train).any())
print("majority-class floor:", max(np.mean(y_test), 1-np.mean(y_test)))

(6954, 26) (1739, 26) False
majority-class floor: 0.503737780333525


In [18]:
import time 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

t0 = time.time()
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
logreg_time = time.time() - t0

logreg_acc = accuracy_score(y_test, logreg.predict(X_test))
print(f"LogReg val accuracy: {logreg_acc:.4f} ({logreg_time:.2f}s)")

LogReg val accuracy: 0.7890 (0.01s)


In [19]:
feat_names = pre.get_feature_names_out()
import pandas as pd
print(pd.Series(logreg.coef_[0], index=feat_names).sort_values(key=abs, ascending=False).head(8).to_string())

num__Spa                 -2.266208
num__VRDeck              -1.958919
cat__Deck_C               1.243273
num__RoomService         -0.994967
cat__HomePlanet_Europa    0.869141
num__FoodCourt            0.786137
cat__Deck_A              -0.728023
cat__CryoSleep_True       0.698224


In [23]:
import torch, torch.nn as nn 

class MLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1),        
        )
    def forward(self, x):
        return self.net(x)


model = MLP(26)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []

for epoch in range(1,31):
    model.train()
    running, n = 0.0, 0
    for xb, yb in train_dl:
        optimizer.zero_grad()

        logits = model(xb).squeeze(1)

        loss = loss_fn(logits, yb)

        loss.backward() 

        optimizer.step()

        running += loss.item() * len(yb) ; n += len(yb)

    model.eval()
    correct, vtotal, vn = 0, 0.0, 0
    with torch.no_grad():
        for xb, yb in test_dl:
            logits = model(xb).squeeze(1)
            vtotal += loss_fn(logits, yb).item()*len(yb)
            correct += ((logits > 0).float() == yb).sum().item()
            vn += len(yb)

    history.append({"epoch": epoch, "train_loss": running/n,
                    "val_loss": vtotal/vn, "val_acc": correct/vn})
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  train {running/n:.4f}  val {vtotal/vn:.4f}  acc {correct/vn:.4f}")


epoch  5  train 0.4214  val 0.4117  acc 0.7964
epoch 10  train 0.4132  val 0.4033  acc 0.7970
epoch 15  train 0.4089  val 0.3978  acc 0.8028
epoch 20  train 0.4005  val 0.3992  acc 0.7993
epoch 25  train 0.3983  val 0.3920  acc 0.8068
epoch 30  train 0.3948  val 0.3948  acc 0.8062


In [24]:
import numpy as np, pandas as pd
mlp_acc = max(h["val_acc"] for h in history)
print(f"LogReg {logreg_acc:.4f} | MLP {mlp_acc:.4f} | delta {mlp_acc-logreg_acc:+.4f} "
      f"({(mlp_acc-logreg_acc)*1739:+.0f} passengers)")

# does the gap survive a different split?
from sklearn.model_selection import train_test_split
for seed in [0, 1, 2, 3, 4]:
    Xa, Xb, ya, yb_ = train_test_split(np.vstack([X_train, X_test]),
                                       np.concatenate([y_train, y_test]),
                                       test_size=0.2, random_state=seed)
    acc = LogisticRegression(max_iter=1000).fit(Xa, ya).score(Xb, yb_)
    print(f"  seed {seed}: LogReg {acc:.4f}")

LogReg 0.7890 | MLP 0.8097 | delta +0.0207 (+36 passengers)
  seed 0: LogReg 0.7982
  seed 1: LogReg 0.7838
  seed 2: LogReg 0.7803
  seed 3: LogReg 0.7855
  seed 4: LogReg 0.7895


On 6954 rows and 26 features, a 2-hidden-layer MLP beat logistic regression by roughly 1–2 accuracy points — the same order of magnitude as the variation between random splits, and measured with best-epoch selection that flatters the MLP. I'd need repeated runs across seeds for both models before calling it a real difference.

The answer

The MLP is the more flexible model — stacked hidden layers with non-linearities fit shapes a single linear boundary can't — and dropout buys back some of the overfitting risk that flexibility creates. That's what the baseline is for: to find out whether the flexibility actually paid. Here it bought 1–2 points, the same size as the variation between random splits, so on this dataset I can't claim the neural net is better at all.

The mistake would be treating "linear vs. flexible" as the axis that matters. Gradient boosted trees are just as flexible as my MLP and usually win on tables. The real difference is inductive bias — the assumptions a model makes before seeing data. A tree splits one feature at a time (TotalSpend > 500) in one exact operation; my MLP approximates that same step with a weighted sum over all 26 inputs and a stack of ReLUs. Trees also ignore scale and skew — the standardising and log1p I did yesterday was work a tree wouldn't need — and handle mixed types natively instead of exploding ~12 real features into 26 one-hot columns. A CNN wins on images because convolution's bias (nearby pixels are related) matches the data; nothing about an MLP's bias matches a table, which is why it needs more data to reach the same place.

So a neural net earns its place on tabular data only when the table stops behaving like a plain table:

High-cardinality categoricals — 50,000 product or tool IDs, where one-hot is hopeless and learned embeddings are the right answer.
Mixed modality — a column is free text, or a path to a wafer image. One differentiable pipeline absorbs it; a tree needs a second model bolted alongside. Most likely case at TSMC.
Scale — millions of rows, where data hunger stops being a liability.
Multi-task / transfer — many related predictions over one schema, sharing a trunk.
End-to-end differentiability — the table feeds a larger neural system and gradients must flow through it.
One sentence

On a small tabular problem, use logistic regression to set the floor and gradient boosting to win; reach for a neural net when the table carries something a tree can't represent — huge categorical vocabularies, another modality, or a downstream network that needs gradients.

Things I noticed that weren't in the plan
Val loss below train loss at nearly every epoch. Not a bug — train loss is measured with dropout active (30% of neurons off), val loss with model.eval() and the full network. Handicapped model vs. healthy one.
Never overfit — both losses still falling at epoch 30. I stopped because the loop said 30, not because the curves told me to.
loss.backward() belongs to the loss, not the optimizer. Backward walks the graph and fills p.grad; optimizer.step() only applies what's already there; zero_grad() is needed because PyTorch accumulates into .grad instead of overwriting — forget it and nothing crashes, the numbers just quietly go wrong.
Still unsure
Never ran gradient boosting, so "it beats both" is borrowed, not measured. Four lines of XGBoost would close that.
Ran the MLP once, on one split. A fair comparison repeats both models across seeds and compares distributions, not single numbers.

In [1]:
"""
LeetCode 199 - scratchpad.
Run this and READ THE OUTPUT. The traversal is instrumented so you can
watch the queue fill from the right and drain from the left.

Nothing here solves the problem for you - the collecting step is yours.
"""

from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build(values):
    """Build a tree from LeetCode's level-order list, e.g. [1,2,3,None,5,None,4]."""
    if not values:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def trace(root):
    """Plain level-order BFS with printing. Watch the queue."""
    if not root:
        print("empty tree")
        return

    q = deque([root])
    level = 0

    while q:
        n = len(q)
        print(f"\n--- level {level} --- frozen n = {n}   queue: {[x.val for x in q]}")

        for i in range(n):
            a = q.popleft()

            if a.left:
                q.append(a.left)
            if a.right:
                q.append(a.right)

            tag = "  <-- LAST of this level (rightmost)" if i == n - 1 else ""
            print(f"   i={i}  popped {a.val:<3} queue now: {[x.val for x in q]}{tag}")

        level += 1

    print("\nqueue empty -> done")


if __name__ == "__main__":
    # LeetCode example 1. Expected right side view: [1, 3, 4]
    print("=" * 60)
    print("TREE A:  [1,2,3,null,5,null,4]")
    print("""
        1
       / \\
      2   3
       \\   \\
        5   4
    """)
    trace(build([1, 2, 3, None, 5, None, 4]))

    # The trap case: rightmost branch is SHORT.
    # Level 2's only node is 5, reached via the LEFT child.
    # Expected right side view: [1, 3, 5]
    print("\n" + "=" * 60)
    print("TREE B:  [1,2,3,null,5]   <- the case that breaks naive solutions")
    print("""
        1
       / \\
      2   3
       \\
        5
    """)
    trace(build([1, 2, 3, None, 5]))

    # TODO (yours):
    #   Copy trace() into a new function rightSideView(root).
    #   Delete every print.
    #   Add a result list, and append the right thing at the right moment.
    #   Hint you already have: the moment is i == n - 1, and TreeNode
    #   is not what the answer list is made of.

TREE A:  [1,2,3,null,5,null,4]

        1
       / \
      2   3
       \   \
        5   4
    

--- level 0 --- frozen n = 1   queue: [1]
   i=0  popped 1   queue now: [2, 3]  <-- LAST of this level (rightmost)

--- level 1 --- frozen n = 2   queue: [2, 3]
   i=0  popped 2   queue now: [3, 5]
   i=1  popped 3   queue now: [5, 4]  <-- LAST of this level (rightmost)

--- level 2 --- frozen n = 2   queue: [5, 4]
   i=0  popped 5   queue now: [4]
   i=1  popped 4   queue now: []  <-- LAST of this level (rightmost)

queue empty -> done

TREE B:  [1,2,3,null,5]   <- the case that breaks naive solutions

        1
       / \
      2   3
       \
        5
    

--- level 0 --- frozen n = 1   queue: [1]
   i=0  popped 1   queue now: [2, 3]  <-- LAST of this level (rightmost)

--- level 1 --- frozen n = 2   queue: [2, 3]
   i=0  popped 2   queue now: [3, 5]
   i=1  popped 3   queue now: [5]  <-- LAST of this level (rightmost)

--- level 2 --- frozen n = 1   queue: [5]
   i=0  popped 5   queue